In [40]:
import numpy as np
import pandas as pd
import pwlf
from datetime import timedelta
# === 参数 ===
excel_file = '../data/Source.xlsx'  # Excel 文件名（与脚本放同目录）
# 设定目标日期
target_date = pd.Timestamp('2025-04-19')
start_date = target_date - timedelta(days=10)
n_segments = 3  # CoPiLinear 拟合段数
# === 读取数据 ===
df = pd.read_excel(excel_file, usecols=[0, 1, 2, 9, 10])
df['日期'] = pd.to_datetime(df['日期'])

In [42]:
# === 拟合模型 ===
# 筛选日期在 start_date（含）到 target_date（不含）之间的数据
fit_df = df[(df['日期'] >= start_date) & (df['日期'] < target_date)]
x = fit_df['日前负荷率(%)'].values
y = fit_df['(调控后)日前出清价格(元/MWh)'].values

# 定义权重：同样对 x<=0.3 和 x>0.7 数据赋予较大权重
weights = np.ones_like(x)
weights[x <= 0.3] = 3.0
weights[x > 0.8] = 3.0

# 初始参数猜测
model = pwlf.PiecewiseLinFit(x, y, weights=weights)
# 拟合3段线性函数（不需要在 fit 中再传 weights 参数）
breaks = model.fit(n_segments)  

In [ ]:
# === 数据清洗 ===
processing_df = df[df['日期'] == target_date].copy()
x_processing = processing_df['日前负荷率(%)'].values
y_processing = processing_df['(调控后)日前出清价格(元/MWh)'].values
y_pred = model.predict(x_processing)
# === 异常值检测（基于残差和 IQR） ===
residuals = y_processing - y_pred
Q1 = np.percentile(residuals, 25)
Q3 = np.percentile(residuals, 75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

processing_df['(调控后)日前出清价格(元/MWh)'] = y_pred
processing_df['residual'] = residuals
processing_df['is_outlier'] = (residuals < lower_bound) | (residuals > upper_bound)

processing_df[processing_df['is_outlier']]